In [1]:
import os
os.getcwd()

'C:\\Users\\PC'

In [2]:
os.chdir(r"D:\FQL\PJ 5")

## Section 1 : Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance

In [6]:
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded')

Libraries loaded


## Section 2 : Load Data

In [7]:
df = pd.read_csv('osteoporosis.csv')
df.drop(columns=['ID'], inplace=True, errors='ignore')

In [8]:
print(f'Shape: {df.shape}')
df.head()

Shape: (1958, 16)


,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
0,104866,69,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,Yes,Moderate,Rheumatoid Arthritis,Corticosteroids,Yes,1
1,101999,32,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,No,NaN,NaN,NaN,Yes,1
2,106567,89,Female,Postmenopausal,No,Caucasian,Normal,Adequate,Sufficient,Active,No,Moderate,Hyperthyroidism,Corticosteroids,No,1
3,102316,78,Female,Normal,No,Caucasian,Underweight,Adequate,Insufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,Corticosteroids,No,1
4,101944,38,Male,Postmenopausal,Yes,African American,Normal,Low,Sufficient,Active,Yes,NaN,Rheumatoid Arthritis,NaN,Yes,1


## Section 3 : Preprocessing

#### ── 3.1 Handle Missing Values ──────────────────────────────────────

In [9]:
print('Missing values before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before:
Alcohol Consumption    988
Medical Conditions     647
Medications            985
dtype: int64


In [10]:
# Fill numerical with median
for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical with mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [11]:
print('\nMissing values after:', df.isnull().sum().sum())


Missing values after: 0


#### ── 3.2 Label Encoding for Binary/Ordinal Features ──────────────────

In [12]:
# Binary mappings
binary_map = {'Yes': 1, 'No': 0}
binary_cols = ['Family History', 'Smoking', 'Alcohol Consumption', 'Prior Fractures']

In [13]:
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Ordinal mappings
df['Hormonal Changes']  = df['Hormonal Changes'].map({'Normal': 0, 'Postmenopausal': 1})
df['Body Weight']       = df['Body Weight'].map({'Underweight': 0, 'Normal': 1, 'Overweight': 2})
df['Calcium Intake']    = df['Calcium Intake'].map({'Low': 0, 'Adequate': 1})
df['Vitamin D Intake']  = df['Vitamin D Intake'].map({'Insufficient': 0, 'Sufficient': 1})
df['Physical Activity'] = df['Physical Activity'].map({'Sedentary': 0, 'Moderate': 1, 'Active': 2})
df['Gender']            = df['Gender'].map({'Female': 1, 'Male': 0})

# One-Hot Encoding for nominal features
df = pd.get_dummies(df, columns=['Race/Ethnicity', 'Medical Conditions', 'Medications'], drop_first=True)

# Encode target
df['Osteoporosis'] = df['Osteoporosis'].map({'Yes': 1, 'No': 0})

print('Encoding complete')

Encoding complete


In [14]:
print(f'Shape after encoding: {df.shape}')
df.head()

Shape after encoding: (1958, 16)


,Id,Age,Gender,Hormonal Changes,Family History,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Prior Fractures,Osteoporosis,Race/Ethnicity_Asian,Race/Ethnicity_Caucasian,Medical Conditions_Rheumatoid Arthritis
0,104866,69,1,0,1,0,0,1,0,1,NaN,1,NaN,True,False,True
1,101999,32,1,0,1,0,0,1,0,0,NaN,1,NaN,True,False,False
2,106567,89,1,1,0,1,1,1,2,0,NaN,0,NaN,False,True,False
3,102316,78,1,0,0,0,1,0,0,1,NaN,0,NaN,False,True,True
4,101944,38,0,1,1,1,0,1,2,1,NaN,1,NaN,False,False,True


## Section 4 : Feature Engineering 

#### ── 4.1 Feature Engineering ──────────────────────────────────────────

In [16]:
df['Nutrient_Deficiency'] = ((df['Calcium Intake'] == 0) | (df['Vitamin D Intake'] == 0)).astype(int)
df['Lifestyle_Risk']      = df['Smoking'] + df['Alcohol Consumption']
df['Age_Group']           = pd.cut(df['Age'], bins=[0, 40, 55, 70, 120],
                                   labels=[0, 1, 2, 3]).astype(int)
df['Hormonal_Bone_Risk']  = df['Hormonal Changes'] * df['Family History']

print('Engineered features added')
print('New features: Nutrient_Deficiency, Lifestyle_Risk, Age_Group, Hormonal_Bone_Risk')

Engineered features added
New features: Nutrient_Deficiency, Lifestyle_Risk, Age_Group, Hormonal_Bone_Risk


#### ── 4.2 Prepare X, y ─────────────────────────────────────────────────

In [17]:
X = df.drop(columns=['Osteoporosis'])
y = df['Osteoporosis']

print(f'Features (X): {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Target (y) distribution:\n{y.value_counts()}')

Features (X): 19 | Samples: 1958
Target (y) distribution:
Series([], Name: count, dtype: int64)
